# Ordinary Differential Equations, Variational Calculus, and Hamiltonian Mechanics

This tutorial demonstrates using **`symlie`** for:
1. **Ordinary Differential Equations (ODEs)**: Symmetry analysis of the Linear Harmonic Oscillator $\ddot{y} + y = 0$ and the Scale-Invariant ODE $y' = y/t$.
2. **Variational Calculus**: Deriving equations of motion via the **Euler–Lagrange operator** `euler_lagrange`.
3. **Canonical Hamiltonian Mechanics**: Computing Poisson brackets `poisson_bracket` and constructing Hamilton's equations `hamilton_equations`.

In [ ]:
import sympy as sp

from symlie import (
    euler_lagrange,
    hamilton_equations,
    infinitesimals,
    poisson_bracket,
    verify_generator,
)

sp.init_printing()

## 1. Symmetry Analysis of Ordinary Differential Equations

### A. The Linear Harmonic Oscillator (2nd-Order ODE)
$$\ddot{y}(t) + y(t) = 0$$


Within the total-degree-1 polynomial ansatz, the equation admits a 2-dimensional subalgebra (time translation and a scaling symmetry). This is not the complete point-symmetry algebra: by Lie's classical theorem, any linear 2nd-order ODE admits the full 8-dimensional $\mathfrak{sl}(3, \mathbb{R})$ algebra, whose remaining generators are quadratic in $y$ and lie outside this degree-1 ansatz.

In [ ]:
t = sp.symbols("t")
y = sp.Function("y")(t)
osc_ode = y.diff(t, 2) + y

sol_osc = infinitesimals(osc_ode, y, t, ansatz_degree=1)
print(
    f"Harmonic oscillator dimension within degree-1 ansatz: {sol_osc.ansatz_dimension}"
)
for i, gen in enumerate(sol_osc.basis, 1):
    print(
        f"X_{i}: xi^t = {gen.xi[0]},  phi^y = {gen.phi[0]}  |  Valid: {verify_generator(osc_ode, y, t, gen)}"
    )

### B. Scale-Invariant First-Order ODE
$$y'(t) - \frac{y}{t} = 0$$

Within the total-degree-1 polynomial ansatz, the equation admits a 4-dimensional $\mathfrak{gl}(2, \mathbb{R})$ subalgebra. This is not a claim that the complete point-symmetry algebra of the first-order ODE is four-dimensional.


The automatically inferred substitution rule for $y'$ requires $t \neq 0$ (already a singular point of the equation's own coefficients), reported via `sol_scale.regularity_conditions` below.

In [ ]:
scale_ode = y.diff(t) - y / t
sol_scale = infinitesimals(scale_ode, y, t, ansatz_degree=1)
print(f"Scale-Invariant ODE Symmetries: {sol_scale.dimension}")
for i, gen in enumerate(sol_scale.basis, 1):
    print(
        f"X_{i}: xi^t = {gen.xi[0]},  phi^y = {gen.phi[0]}  |  Valid: {verify_generator(scale_ode, y, t, gen)}"
    )

print(f"Regularity conditions: {sol_scale.regularity_conditions}")

## 2. Variational Mechanics: The Euler–Lagrange Operator

For the Harmonic Oscillator with mass $m$ and spring constant $k$:
$$\mathcal{L}(t, q, \dot{q}) = \frac{1}{2} m \dot{q}^2 - \frac{1}{2} k q^2$$

The Euler–Lagrange operator evaluates:
$$\mathbf{E}_q(\mathcal{L}) = \frac{\partial \mathcal{L}}{\partial q} - \frac{d}{dt}\left(\frac{\partial \mathcal{L}}{\partial \dot{q}}\right) = -kq - m\ddot{q}$$

In [ ]:
q = sp.Function("q")(t)
m, k = sp.symbols("m k", positive=True)
L = sp.Rational(1, 2) * m * q.diff(t) ** 2 - sp.Rational(1, 2) * k * q**2

el_res = euler_lagrange(L, q, t)
print("Euler-Lagrange Equation E_q(L) = 0:")
display(sp.Eq(el_res[0], 0))

## 3. Canonical Hamiltonian Dynamics & Poisson Brackets

In phase space with canonical coordinate $q$ and momentum $p$, the Hamiltonian is:
$$H(q, p) = \frac{p^2}{2m} + \frac{1}{2} k q^2$$

Canonical Poisson brackets satisfy:
$$\{q, p\} = 1, \quad \{p, q\} = -1, \quad \{q, q\} = 0$$

Time evolution of observables $f$:
$$\dot{q} = \{q, H\} = \frac{p}{m}, \qquad \dot{p} = \{p, H\} = -kq$$

In [ ]:
qt = sp.Function("q")(t)
pt = sp.Function("p")(t)
H = pt**2 / (2 * m) + sp.Rational(1, 2) * k * qt**2

print("Canonical Brackets:")
print(" {q, p} =", poisson_bracket(qt, pt, qt, pt))
print(" {p, q} =", poisson_bracket(pt, qt, qt, pt))

print("\nEquations of motion from Poisson bracket:")
print(" dq/dt = {q, H} =", poisson_bracket(qt, H, qt, pt))
print(" dp/dt = {p, H} =", poisson_bracket(pt, H, qt, pt))

print("\nHamilton's Equations of Motion (direct):")
eqs = hamilton_equations(H, qt, pt, time=t)
for eq in eqs:
    display(eq)